#### <b> Import Statements & Data Reading </b> 

In [22]:
import numpy as np
import pandas as pd

match_data = pd.read_excel("epl_match_data.xlsx", sheet_name = "Match Data")
season_results = pd.read_excel("epl_match_data.xlsx", sheet_name = "Season Results Data")
transfer_data = pd.read_excel("epl_transfer_data.xlsx")

In [ ]:
# Define constants to support data processing procedures
STARTING_YR = 2015
ENDING_YR = 2026

PREV_MATCHES = 10 # set this constant to determine the number of matches from the previous season to consider for predicting end of season results
CURR_MATCHES = 10 # set this constant to determine the number of matches from the current season to consider for predicting end of season results

#### <b> Data Standardization </b>
To ensure consistency across the match and transfer datasets, club names were standardized using a mapping dictionary to account for differences in naming conventions. The resulting club names were validated to confirm that both datasets contained the same set of teams. Additionally, season columns were converted to integer data types to ensure consistency when merging records across datasets.

In [24]:
# Standardizing team names across datasets
# Create a mapping of team names to standardized names
club_name_map = {
    "AFC Bournemouth": "Bournemouth",
    "Arsenal FC": "Arsenal",
    "Brentford FC": "Brentford",
    "Brighton & Hove Albion": "Brighton",
    "Burnley FC": "Burnley",
    "Cardiff City": "Cardiff",
    "Chelsea FC": "Chelsea",
    "Everton FC": "Everton",
    "Fulham FC": "Fulham",
    "Huddersfield Town": "Huddersfield",
    "Hull City": "Hull",
    "Ipswich Town": "Ipswich",
    "Leeds United": "Leeds",
    "Leicester City": "Leicester",
    "Liverpool FC": "Liverpool",
    "Luton Town": "Luton",
    "Middlesbrough FC": "Middlesbrough",
    "Norwich City": "Norwich",
    "Southampton FC": "Southampton",
    "Stoke City": "Stoke",
    "Sunderland AFC": "Sunderland",
    "Swansea City": "Swansea",
    "Tottenham Hotspur": "Tottenham",
    "Watford FC": "Watford",
    "West Ham United": "West Ham"
}

transfer_data["club"] = transfer_data["club"].replace(club_name_map) # Standardize club names in transfer_data using the mapping
len(transfer_data["club"].unique()) == len(match_data["Team"].unique()) # Ensure number of unique clubs is the same in both datasets
set(transfer_data["club"].unique()) == set(match_data["Team"].unique()) # Ensure the unique clubs are the same in both datasets

transfer_data["season"] = transfer_data["season"].astype(int) # Convert the "season" column in transfer_data to integer type
match_data["Season"] = match_data["Season"].astype(int) # Convert the "Season" column in match_data to integer type

#### <b> Feature Engineering from Transfer Data </b>
Transfer records were aggregated by season, club, and transfer window to create features describing each team's transfer activity. These included total transfer spending, transfer earnings, net spending, counts of incoming and outgoing permanent transfers and loans, and the average age of incoming players. Additional variables were created to count transfers with missing fee information. The resulting statistics were merged into a single DataFrame, with missing aggregate values filled with zero. Only summer transfer statistics were retained for the final modeling dataset to ensure that all predictors were available before the Match 10 prediction point.

In [25]:
window_spending = transfer_data[transfer_data["movement"] == "in"].groupby(["season", "club", "window"]).agg({"fee": "sum"}).reset_index() # Calculate the total transfer fees for incoming transfers by season, club, and window
window_earnings = transfer_data[(transfer_data["movement"] == "out")].groupby(["season", "club", "window"]).agg({"fee": "sum"}).reset_index() # Calculate the total estimated transfer earnings for outgoing transfers by season, club, and window

net_spending = pd.merge(window_spending, window_earnings, on = ["season", "club", "window"],  how = "outer", suffixes = ("_spending", "_earnings")) # Merge spending and earnings data
net_spending["net_spending"] = net_spending["fee_spending"].fillna(0) - net_spending["fee_earnings"].fillna(0) # Calculate net spending by subtracting earnings from spending, filling NaN values with 0

permanent_in = transfer_data[(transfer_data["movement"] == "in") & (transfer_data["is_loan"] == False)].groupby(["season", "club", "window"]).agg({"player_name": "count"}).reset_index() # Count the number of incoming transfers by season, club, and window
permanent_out = transfer_data[(transfer_data["movement"] == "out") & (transfer_data["is_loan"] == False)].groupby(["season", "club", "window"]).agg({"player_name": "count"}).reset_index() # Count the number of outgoing transfers by season, club, and window
permanent_in.rename(columns={"player_name": "permanent_in_count"}, inplace=True) # Rename the column for incoming transfers count
permanent_out.rename(columns={"player_name": "permanent_out_count"}, inplace=True) # Rename the column for outgoing transfers count

loan_in = transfer_data[(transfer_data["movement"] == "in") & (transfer_data["is_loan"] == True)].groupby(["season", "club", "window"]).agg({"player_name": "count"}).reset_index() # Count the number of incoming loans by season, club, and window
loan_out = transfer_data[(transfer_data["movement"] == "out") & (transfer_data["is_loan"] == True)].groupby(["season", "club", "window"]).agg({"player_name": "count"}).reset_index() # Count the number of outgoing loans by season, club, and window
loan_in.rename(columns={"player_name": "loan_in_count"}, inplace=True) # Rename the column for incoming loans count
loan_out.rename(columns={"player_name": "loan_out_count"}, inplace=True) # Rename the column for outgoing loans count

average_age_in = transfer_data[transfer_data["movement"] == "in"].groupby(["season", "club", "window"]).agg({"age": "mean"}).reset_index() # Calculate the average age of incoming transfers by season, club, and window
average_age_in.rename(columns = {"age": "average_age_in"}, inplace = True) # Rename the column for average age of incoming transfers

in_missing_fees = transfer_data[(transfer_data["movement"] == "in") & (transfer_data["fee"].isnull())].groupby(["season", "club", "window"]).size().reset_index(name="missing_in_fees_count") # Count the number of incoming transfers with missing fee information by season, club, and window
out_missing_fees = transfer_data[(transfer_data["movement"] == "out") & (transfer_data["fee"].isnull())].groupby(["season", "club", "window"]).size().reset_index(name="missing_out_fees_count") # Count the number of outgoing transfers with missing fee information by season, club, and window

In [26]:
# Merging all the transfer-related data into a single DataFrame
transfer_stats = net_spending.copy() # Start with net spending data
for df in [permanent_in, permanent_out, loan_in, loan_out, average_age_in, in_missing_fees, out_missing_fees]:
    transfer_stats = pd.merge(transfer_stats, df, on=["season", "club", "window"], how="outer", suffixes=('', '_x')) # Merge each DataFrame into transfer_stats on season, club, and window
transfer_stats = transfer_stats.fillna(0) # Fill NaN values in the merged transfer statistics DataFrame with 0 
transfer_stats

,season,club,window,fee_spending,fee_earnings,net_spending,permanent_in_count,permanent_out_count,loan_in_count,loan_out_count,average_age_in,missing_in_fees_count,missing_out_fees_count
0,2015,Arsenal,summer,14000000.0,2500000.0,11500000.0,1.0,4.0,0.0,6.0,33.000000,0.0,0.0
1,2015,Arsenal,winter,12500000.0,0.0,12500000.0,1.0,0.0,0.0,2.0,23.000000,0.0,0.0
2,2015,Aston Villa,summer,66550000.0,64700000.0,1850000.0,11.0,14.0,1.0,10.0,25.166667,0.0,2.0
3,2015,Aston Villa,winter,0.0,0.0,0.0,0.0,2.0,0.0,6.0,0.000000,0.0,0.0
4,2015,Bournemouth,summer,31000000.0,0.0,31000000.0,9.0,10.0,3.0,10.0,26.416667,0.0,6.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
414,2025,Nottingham Forest,summer,236900000.0,126900000.0,110000000.0,11.0,13.0,2.0,7.0,24.153846,0.0,3.0
415,2025,Sunderland,summer,187900000.0,51000000.0,136900000.0,13.0,5.0,2.0,11.0,25.066667,0.0,1.0
416,2025,Tottenham,summer,210600000.0,42500000.0,168100000.0,6.0,6.0,2.0,9.0,23.250000,0.0,3.0
417,2025,West Ham,summer,143800000.0,89500000.0,54300000.0,8.0,9.0,1.0,7.0,26.666667,1.0,1.0


#### <b> Preparing Data for Model Development </b>

Created the following pandas DataFrame that combines historical match performance, end-of-season results, and summer transfer activity for English Premier League clubs. The dataset contains 200 club-season observations across 10 seasons, from 2016/17 through 2025/26. The DataFrame is titled `final_merged_data`.

Match performance features include statistics from the first 10 matches of the current season and the final 10 matches of the previous season. Transfer features reflect activity during the summer transfer window preceding the current season. The dataset is intended to support predicting each club's final league points after Match 10.

Below are the columns included in `final_merged_data` along with their definitions.
1. Season — starting year of the Premier League season.
2. Team — name of the Premier League club.
3. Curr_Avg_Win_Percentage — average estimated probability of winning across the first 10 matches of the current season.
4. Curr_Avg_Draw_Percentage — average estimated probability of drawing across the first 10 matches of the current season.
5. Curr_Avg_Loss_Percentage — average estimated probability of losing across the first 10 matches of the current season.
6. Curr_Avg_Goals — average goals scored per match during the first 10 matches of the current season.
7. Curr_Avg_Shots — average shots attempted per match during the first 10 matches of the current season.
8. Curr_Avg_SOT — average shots on target per match during the first 10 matches of the current season.
9. Curr_Avg_Opp_Goals — average goals conceded per match during the first 10 matches of the current season.
10. Curr_Avg_Opp_Shots — average opponent shots per match during the first 10 matches of the current season.
11. Curr_Avg_Opp_SOT — average opponent shots on target per match during the first 10 matches of the current season.
12. Curr_Avg_xG — average expected goals per match during the first 10 matches of the current season.
13. Curr_Avg_xGA — average expected goals against per match during the first 10 matches of the current season.
14. Curr_Avg_npxG — average non-penalty expected goals per match during the first 10 matches of the current season.
15. Curr_Avg_npxGA — average non-penalty expected goals against per match during the first 10 matches of the current season.
16. Curr_Avg_Deep — average deep completions, or passes completed near the opponent's goal, per match during the first 10 matches of the current season.
17. Curr_Avg_Deep_Allowed — average opponent deep completions allowed per match during the first 10 matches of the current season.
18. Curr_Expected_Points — total expected points accumulated across the first 10 matches of the current season.
19. Curr_Points — total actual match points earned during the first 10 matches of the current season.
20. Prev_Total_Points — official final league points earned by returning clubs in the previous season, with newly promoted clubs assigned the previous season's league-average points.
21. EOS_Total_Points — official final league points earned by the club in the current season; the target variable for regression analysis.
22. Prev_Avg_Win_Percentage — average estimated probability of winning across the final 10 matches of the previous season.
23. Prev_Avg_Draw_Percentage — average estimated probability of drawing across the final 10 matches of the previous season.
24. Prev_Avg_Loss_Percentage — average estimated probability of losing across the final 10 matches of the previous season.
25. Prev_Avg_Goals — average goals scored per match during the final 10 matches of the previous season.
26. Prev_Avg_Shots — average shots attempted per match during the final 10 matches of the previous season.
27. Prev_Avg_SOT — average shots on target per match during the final 10 matches of the previous season.
28. Prev_Avg_Opp_Goals — average goals conceded per match during the final 10 matches of the previous season.
29. Prev_Avg_Opp_Shots — average opponent shots per match during the final 10 matches of the previous season.
30. Prev_Avg_Opp_SOT — average opponent shots on target per match during the final 10 matches of the previous season.
31. Prev_Avg_xG — average expected goals per match during the final 10 matches of the previous season.
32. Prev_Avg_xGA — average expected goals against per match during the final 10 matches of the previous season.
33. Prev_Avg_npxG — average non-penalty expected goals per match during the final 10 matches of the previous season.
34. Prev_Avg_npxGA — average non-penalty expected goals against per match during the final 10 matches of the previous season.
35. Prev_Avg_Deep — average deep completions per match during the final 10 matches of the previous season.
36. Prev_Avg_Deep_Allowed — average opponent deep completions allowed per match during the final 10 matches of the previous season.
37. Prev_Expected_Points — total expected points accumulated across the final 10 matches of the previous season.
38. Prev_Points — total actual match points earned during the final 10 matches of the previous season.
39. Promoted — indicator for whether the club was newly promoted to the Premier League, where 1 indicates a promoted club and 0 indicates a returning club.
40. Fee_Spending — total reported transfer and loan fees paid for incoming players during the preceding summer transfer window.
41. Fee_Earnings — total reported transfer and loan fees received for outgoing players during the preceding summer transfer window.
42. Net_Spending — net transfer expenditure during the preceding summer transfer window, calculated as fee spending minus fee earnings.
43. Permanent_In_Count — number of incoming permanent player transfers during the preceding summer transfer window.
44. Permanent_Out_Count — number of outgoing permanent player transfers during the preceding summer transfer window.
45. Loan_In_Count — number of incoming loan transfers during the preceding summer transfer window.
46. Loan_Out_Count — number of outgoing loan transfers during the preceding summer transfer window.
47. Average_Age_In — average age of incoming players during the preceding summer transfer window, with 0 used when no incoming players were recorded.
48. Missing_In_Fees_Count — number of incoming transfers during the preceding summer transfer window with missing fee information.
49. Missing_Out_Fees_Count — number of outgoing transfers during the preceding summer transfer window with missing fee information.


<u> Additional Data Processing Notes </u>

For clubs newly promoted to the Premier League, previous-season match performance statistics were unavailable. These missing values were replaced with the corresponding league averages from the previous season. The Promoted indicator was retained to distinguish these observations from returning clubs.

Transfer spending and earnings reflect the sum of available reported fees. Transfers with missing fee information were excluded from these sums, and separate missing-fee counts were retained to capture uncertainty in the financial variables.

The final dataset contains one observation per club per season, with all predictor variables restricted to information available by the completion of Match 10.

##### Step 1 - Merge match data with transfer statistics 

In [27]:
# Merge all the transfer-related results into a single DataFrame with the match_data
merged_matches_data = pd.DataFrame() # Initialize an empty DataFrame to hold the merged data
for year in range(STARTING_YR, ENDING_YR):
    match_data_year = match_data[match_data["Season"] == year] # Filter match_data for the current year
    transfer_stats_year = transfer_stats[transfer_stats["season"] == year] # Filter transfer_stats for the current year

    # Filter match_data for matches before winter and after winter, merging the corresponding transfer stats for each half of a season then merging with the running merged_matches_data DataFrame
    matches_before_winter = match_data_year[pd.to_datetime(match_data_year["Date"]) < pd.Timestamp(year = year  + 1, month = 1, day = 1)] # Filter match_data for matches before January 1st of the current year
    summer_stats = transfer_stats_year[transfer_stats_year["window"] == "summer"] # Get summer transfer stats for the current year
    matches_before_winter = pd.merge(matches_before_winter, summer_stats, left_on=["Season", "Team"], right_on=["season", "club"], how="left") # Merge match_before_winter with summer_stats on season and club
    
    matches_after_winter = match_data_year[pd.to_datetime(match_data_year["Date"]) >= pd.Timestamp(year = year + 1, month = 1, day = 1)] # Filter match_data for matches on or after January 1st of the current year
    winter_stats = transfer_stats_year[transfer_stats_year["window"] == "winter"] # Get winter transfer stats for the current year
    matches_after_winter = pd.merge(matches_after_winter, winter_stats, left_on=["Season", "Team"], right_on=["season", "club"], how="left") # Merge match_after_winter with winter_stats on season and club

    merged_data_year = pd.concat([matches_before_winter, matches_after_winter], ignore_index=True)
    merged_matches_data = pd.concat([merged_matches_data, merged_data_year], ignore_index=True)
merged_matches_data.drop(columns = ["season", "club", "window"], inplace = True)

In [28]:
# Note: the matches past the winter transfer window of the 2025 season have no data since our transfer data only 
# goes up to the 2025 summer transfer window. This is expected and will be handled in the modeling phase since
# we only consider transfer activity during the summer transfer windows each year.
# merged_matches_data.columns # transfer window performance metrics included at column index 27
merged_matches_data 

,Team,Team_ID,Match_ID,Opponent,Home_or_Away,Season,Date,Result,Win_Percentage,Loss_Percentage,...,fee_spending,fee_earnings,net_spending,permanent_in_count,permanent_out_count,loan_in_count,loan_out_count,average_age_in,missing_in_fees_count,missing_out_fees_count
0,Aston Villa,71,82,Bournemouth,a,2015,2015-08-08 18:00:00,Win,0.2958,0.3435,...,66550000.0,64700000.0,1850000.0,11.0,14.0,1.0,10.0,25.166667,0.0,2.0
1,Aston Villa,71,91,Manchester United,h,2015,2015-08-14 22:45:00,Loss,0.1056,0.4291,...,66550000.0,64700000.0,1850000.0,11.0,14.0,1.0,10.0,25.166667,0.0,2.0
2,Aston Villa,71,104,Crystal Palace,a,2015,2015-08-22 18:00:00,Loss,0.2174,0.4747,...,66550000.0,64700000.0,1850000.0,11.0,14.0,1.0,10.0,25.166667,0.0,2.0
3,Aston Villa,71,113,Sunderland,h,2015,2015-08-29 18:00:00,Draw,0.8747,0.0248,...,66550000.0,64700000.0,1850000.0,11.0,14.0,1.0,10.0,25.166667,0.0,2.0
4,Aston Villa,71,129,Leicester,a,2015,2015-09-13 19:00:00,Loss,0.2446,0.4550,...,66550000.0,64700000.0,1850000.0,11.0,14.0,1.0,10.0,25.166667,0.0,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8355,Nottingham Forest,249,29115,Sunderland,a,2025,2026-04-24 19:00:00,Win,0.6401,0.1184,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8356,Nottingham Forest,249,29122,Chelsea,a,2025,2026-05-04 15:00:00,Win,0.4074,0.3108,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8357,Nottingham Forest,249,29134,Newcastle United,h,2025,2026-05-10 14:00:00,Draw,0.3112,0.4214,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8358,Nottingham Forest,249,29145,Manchester United,a,2025,2026-05-17 12:30:00,Loss,0.0591,0.8387,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [29]:
# Verify data merging results
transfer_sample = transfer_stats[(transfer_stats["season"] == 2022) & (transfer_stats["club"] == "Crystal Palace") & (transfer_stats["window"] == "summer")]
merged_sample = merged_matches_data[(merged_matches_data["Season"] == 2022) & (merged_matches_data["Team"] == "Crystal Palace")]
merged_sample.iloc[0, 27:] == transfer_sample.iloc[0, 3:] # Check if the transfer-related data in merged_matches_data matches the corresponding data in transfer_stats for Crystal Palace in the summer of 2022
merged_matches_data.groupby(["Season", "Team"]).size() # should equal 220 (20 teams * 11 seasons)
merged_matches_data.groupby(["Season", "Match_ID"]).size().value_counts() # should equal 2 for the 4180 matches across 11 seasons

2    4180
Name: count, dtype: int64

##### Step 2 - Manipulate data for model inputs

In [30]:
# Define helper function to aggregate match performance results over a specified time frame (e.g., 10 matches at the end of the previous season, 10 matches at beginning of current season)
def agg_match_features(data, prefix):
    return (
        data.groupby(["Season", "Team"])
        .agg(
            **{
                f"{prefix}_Avg_Win_Percentage": ("Win_Percentage", "mean"),
                f"{prefix}_Avg_Draw_Percentage": ("Draw_Percentage", "mean"),
                f"{prefix}_Avg_Loss_Percentage": ("Loss_Percentage", "mean"),
                f"{prefix}_Avg_Goals": ("Goals", "mean"),
                f"{prefix}_Avg_Shots": ("Shots", "mean"),
                f"{prefix}_Avg_SOT": ("Shots_on_Target", "mean"),
                f"{prefix}_Avg_Opp_Goals": ("Opp_Goals", "mean"),
                f"{prefix}_Avg_Opp_Shots": ("Opp_Shots", "mean"),
                f"{prefix}_Avg_Opp_SOT": ("Opp_Shots_on_Target", "mean"),
                f"{prefix}_Avg_xG": ("xG", "mean"),
                f"{prefix}_Avg_xGA": ("xGA", "mean"),
                f"{prefix}_Avg_npxG": ("npxG", "mean"),
                f"{prefix}_Avg_npxGA": ("npxGA", "mean"),
                f"{prefix}_Avg_Deep": ("Deep", "mean"),
                f"{prefix}_Avg_Deep_Allowed": ("Deep_Allowed", "mean"),
                f"{prefix}_Expected_Points": ("Expected_Points", "sum"),
                f"{prefix}_Points": ("Points", "sum"),
            }
        )
        .reset_index()
    )

In [31]:
matches = merged_matches_data.copy() # Crate copy of merged matches with transfer data for manipulation
previous_matches = matches.groupby(["Season", "Team"]).tail(PREV_MATCHES)  # Last 10 matches of every club-season
current_matches = matches[matches["Season"] >= STARTING_YR + 1].groupby(["Season", "Team"]).head(CURR_MATCHES)  # First 10 matches of every club-season

# Generate match performance results for each match window from agg_match_features helper 
prev_prefix = "Prev"
curr_prefix = "Curr"
prev_match_features = agg_match_features(previous_matches, prev_prefix)
curr_match_features = agg_match_features(current_matches, curr_prefix)

# Create a column that represents each team's total points from the previous season
points_lookup = season_results.set_index(["Season", "Team"])["Total_Points"]
curr_match_features["Prev_Total_Points"] = curr_match_features.apply(
    lambda x: points_lookup.get((x["Season"] - 1, x["Team"]), np.nan), axis=1
)

# Create a column that represents each team's total points during the current season, this will represent our repsonse variable for predicting end of season point results for teams based on our predictors
curr_match_features["EOS_Total_Points"] = curr_match_features.apply(
    lambda x: points_lookup.get((x["Season"], x["Team"]), np.nan), axis=1
)

prev_match_features["Merge_Season"] = prev_match_features["Season"] + 1 # Create label to support proper merging of match performance across the prior and current seasons of each observation
season_avg_pts = season_results.groupby("Season")["Total_Points"].mean().reset_index()

match_cols = ['Prev_Avg_Win_Percentage', 'Prev_Avg_Draw_Percentage',
       'Prev_Avg_Loss_Percentage', 'Prev_Avg_Goals', 'Prev_Avg_Shots',
       'Prev_Avg_SOT', 'Prev_Avg_Opp_Goals', 'Prev_Avg_Opp_Shots',
       'Prev_Avg_Opp_SOT', 'Prev_Avg_xG', 'Prev_Avg_xGA', 'Prev_Avg_npxG',
       'Prev_Avg_npxGA', 'Prev_Avg_Deep', 'Prev_Avg_Deep_Allowed',
       'Prev_Expected_Points', 'Prev_Points']
prev_match_avg_performance = prev_match_features.groupby("Season")[match_cols].mean() # Compute the average match performance in terms of the specified columns to use a replacements for recently promoted teams who will have no previous season match data 

In [32]:
# Merge the current season aggregated match performance with the previous season aggregated match performance, both in terms of 10 matches (adjust constants at top of notebook to modify)
merged_season_data = pd.merge(
    curr_match_features,
    prev_match_features,
    left_on=["Season", "Team"],
    right_on=["Merge_Season", "Team"],
    how="left",
)

# Fill empty entries for recently promoted team's previous season match performance with mean results computed as prev_match_avg_performance
merged_season_data["Promoted"] = np.where(merged_season_data["Prev_Points"].isna(), 1, 0) # Assign promoted label for sides promoted to the EPL for that season
for col in match_cols:
    merged_season_data[col] = merged_season_data[col].fillna(
        (merged_season_data["Season_x"] - 1).map(
            prev_match_avg_performance[col]
        )
    )
merged_season_data["Prev_Total_Points"] = merged_season_data["Prev_Total_Points"].fillna(
    (merged_season_data["Season_x"] - 1).map(
        season_avg_pts.set_index("Season")["Total_Points"]
    )
)

transfers = transfer_stats.copy()
transfers = transfers[transfers["window"] == "summer"]  # Isolate transfer results for each summer transfer window for proper merging
transfers.drop(columns="window", inplace=True)

# Merge transfer data for each summer transfer window with the merged match performance data, resulting in observations that each represent 
# a team's average performance across the last 10 matches of the previous season, their average performance across the first 10 matches of the 
# current season, and their transfer window activity during the summer window in between the last season and current season. 
final_merged_data = pd.merge(merged_season_data, transfers, left_on = ["Season_x", "Team"], right_on = ["season", "club"], how = "left")

removed_cols = ["season", "club", "Merge_Season", "Season_y"]
final_merged_data.drop(columns = removed_cols, inplace = True)
final_merged_data.rename(columns={"Season_x": "Season"}, inplace = True)
final_merged_data.columns = list(final_merged_data.columns[:len(merged_season_data.columns) - len(removed_cols)]) + list(final_merged_data.columns[(len(merged_season_data.columns) - len(removed_cols)):].str.title())
final_merged_data

,Season,Team,Curr_Avg_Win_Percentage,Curr_Avg_Draw_Percentage,Curr_Avg_Loss_Percentage,Curr_Avg_Goals,Curr_Avg_Shots,Curr_Avg_SOT,Curr_Avg_Opp_Goals,Curr_Avg_Opp_Shots,...,Fee_Spending,Fee_Earnings,Net_Spending,Permanent_In_Count,Permanent_Out_Count,Loan_In_Count,Loan_Out_Count,Average_Age_In,Missing_In_Fees_Count,Missing_Out_Fees_Count
0,2016,Arsenal,0.56898,0.22699,0.20403,2.3,15.2,5.4,1.0,10.1,...,113000000.0,10350000.0,102650000.0,5.0,5.0,0.0,4.0,23.000000,0.0,2.0
1,2016,Bournemouth,0.25081,0.21708,0.53211,1.2,11.7,3.6,1.4,15.1,...,39750000.0,20190000.0,19560000.0,5.0,14.0,2.0,11.0,22.000000,0.0,4.0
2,2016,Burnley,0.11738,0.23256,0.65006,0.8,8.5,1.9,1.3,21.3,...,24700000.0,0.0,24700000.0,4.0,7.0,2.0,11.0,24.333333,0.0,1.0
3,2016,Chelsea,0.63487,0.19741,0.16772,2.1,16.5,6.1,0.9,9.1,...,132800000.0,42000000.0,90800000.0,5.0,5.0,0.0,20.0,26.800000,1.0,1.0
4,2016,Crystal Palace,0.37812,0.23705,0.38483,1.4,14.7,4.5,1.6,13.5,...,59500000.0,50200000.0,9300000.0,6.0,11.0,1.0,8.0,27.000000,2.0,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,2025,Nottingham Forest,0.24250,0.23140,0.52610,0.7,11.6,3.7,1.9,13.5,...,236900000.0,126900000.0,110000000.0,11.0,13.0,2.0,7.0,24.153846,0.0,3.0
196,2025,Sunderland,0.33830,0.29360,0.36810,1.2,10.5,2.8,0.8,12.9,...,187900000.0,51000000.0,136900000.0,13.0,5.0,2.0,11.0,25.066667,0.0,1.0
197,2025,Tottenham,0.35323,0.22887,0.41790,1.7,9.6,3.4,0.8,12.2,...,210600000.0,42500000.0,168100000.0,6.0,6.0,2.0,9.0,23.250000,0.0,3.0
198,2025,West Ham,0.22452,0.18362,0.59186,1.0,9.9,3.8,2.1,14.5,...,143800000.0,89500000.0,54300000.0,8.0,9.0,1.0,7.0,26.666667,1.0,1.0


In [33]:
# Validate merged data results
len(final_merged_data) # should equal 200 (20 teams * 10 season), the first season is now excluded since we don't have any prior season match data for that season
final_merged_data[final_merged_data.isna().any(axis = 1)] # ensure there are no NaN entries in the dataset

# Ensure each team's end-of-season results reflect the results from imported dataset 
for entry in final_merged_data.iterrows():
    season, team, season_points = entry[1]["Season"], entry[1]["Team"], entry[1]["EOS_Total_Points"]
    table_points = season_results[(season_results["Season"] == season) & (season_results["Team"] == team)]["Total_Points"].iloc[0]
    if table_points != season_points:
        raise ValueError("Incorrect Season Results")

#### <b> Data Export </b>
Run the following line to save the premier league match and transfer data to an Excel file on your device.

In [34]:
with pd.ExcelWriter("epl_merged_data.xlsx") as writer:
    final_merged_data.to_excel(writer, index = False)